In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import shapiro
import scipy.stats as stats

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso

In [34]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [35]:
from sklearn.neural_network import MLPRegressor

In [36]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [37]:
import shap

In [38]:
# Load your cleaned dataset
df = pd.read_csv("df_cleaned.csv")
df.head()

,LOCATION,PRICE,ROOMS,BATHROOMS,CAR_PARKS,PROPERTY_BASE,PROPERTY_TYPE,BUILT-UP,FURNISHING _STATUS,PRICE_LOG,BUILT-UP_LOG
0,SETAPAK,200000.0,3.0,2.0,1.0,CONDOMINIUM,INTERMEDIATE,800.0,UNFURNISHED,12.206078,6.685861
1,SETAPAK,210000.0,3.0,2.0,1.0,CONDOMINIUM,INTERMEDIATE,802.0,PARTLY FURNISHED,12.254868,6.688355
2,SETAPAK,215000.0,3.0,2.0,1.0,CONDOMINIUM,CORNER,900.0,PARTLY FURNISHED,12.278398,6.803505
3,SETAPAK,215000.0,3.0,2.0,1.0,CONDOMINIUM,INTERMEDIATE,800.0,UNFURNISHED,12.278398,6.685861
4,SETAPAK,220000.0,3.0,2.0,1.0,CONDOMINIUM,CORNER,832.0,PARTLY FURNISHED,12.301387,6.725034


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21558 entries, 0 to 21557
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   LOCATION            21558 non-null  object 
 1   PRICE               21558 non-null  float64
 2   ROOMS               21558 non-null  float64
 3   BATHROOMS           21558 non-null  float64
 4   CAR_PARKS           21558 non-null  float64
 5   PROPERTY_BASE       21558 non-null  object 
 6   PROPERTY_TYPE       21558 non-null  object 
 7   BUILT-UP            21558 non-null  float64
 8   FURNISHING _STATUS  21558 non-null  object 
 9   PRICE_LOG           21558 non-null  float64
 10  BUILT-UP_LOG        21558 non-null  float64
dtypes: float64(7), object(4)
memory usage: 1.8+ MB


In [10]:
import random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [11]:
# Numeric columns to be scaled
numeric_features = ['ROOMS', 'BATHROOMS', 'CAR_PARKS', 'BUILT-UP_LOG']

# Categorical columns to be one-hot encoded
categorical_features = ['LOCATION', 'PROPERTY_BASE', 'PROPERTY_TYPE', 'FURNISHING _STATUS']

# Define X (predictors) and y (target)
X = df[numeric_features + categorical_features]
y = df['PRICE_LOG']

In [12]:
# Using the SEED guarantees the exact same test rows for every model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED)

In [13]:
# build the ml pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

In [14]:
# BUILD AND TRAIN THE MLR MODEL

mlr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Fit the pipeline on the 70% training data
mlr_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
# 6. EVALUATE ON TEST DATA
y_pred_mlr = mlr_pipeline.predict(X_test)

print("--- MLR Test Performance ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_mlr)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_mlr):.4f}")
print(f"R²:   {r2_score(y_test, y_pred_mlr):.4f}")

# Save the absolute errors for Model A (MLR)
mlr_absolute_errors = np.abs(y_test - y_pred_mlr)

--- MLR Test Performance ---
RMSE: 0.2574
MAE:  0.1971
R²:   0.8658


C:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [18]:
#Get Baseline Errors (You already have this from the MLR code)
mlr_absolute_errors = np.abs(y_test - y_pred_mlr)
mlr_absolute_errors

4024     0.139780
10887    0.129969
6501     0.394053
4449     0.102589
20092    0.128360
           ...   
4125     0.002356
624      0.123416
5083     0.102706
16933    0.131581
8585     0.038511
Name: PRICE_LOG, Length: 6468, dtype: float64

### Train the LASSO 

In [19]:
# 1. Build the base LASSO pipeline
# (Reuses the 'preprocessor' you already built for MLR)
lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(random_state=SEED, max_iter=10000)) # Increased max_iter to ensure convergence
])


In [20]:
# 2. Define the hyperparameter grid
# Note: Use 'model__alpha' to tell the pipeline to pass these values to the 'model' step
param_grid = {
    'model__alpha': np.arange(0.0001, 0.01, 0.0002).tolist()
}

# 3. Set up the 5-Fold Cross-Validation
# scoring='neg_mean_squared_error' ensures it picks the alpha with the lowest MSE
lasso_cv = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=param_grid,
    cv=5, 
    scoring='neg_mean_squared_error',
    n_jobs=-1 # Uses all CPU cores to speed up training
)

In [21]:
# 4. Run the tuning on the 70% Training Set
print("Tuning LASSO hyperparameters using 5-Fold CV...")
lasso_cv.fit(X_train, y_train)

# Print the best alpha found by the grid search
# Extract best alpha and the CV MSE (multiply by -1 to get positive MSE)
best_alpha = lasso_cv.best_params_['model__alpha']
best_cv_mse = -lasso_cv.best_score_
print(f"Best LASSO alpha: {best_alpha}")
print(f"Lowest CV MSE: {best_cv_mse:.4f}\n")

# 5. Evaluate the best refitted model on the 30% Test Set
best_lasso_model = lasso_cv.best_estimator_
y_pred_lasso = best_lasso_model.predict(X_test)

print("\n--- LASSO Test Performance ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_lasso):.4f}")
print(f"R²:   {r2_score(y_test, y_pred_lasso):.4f}")



Tuning LASSO hyperparameters using 5-Fold CV...
Best LASSO alpha: 0.0001
Lowest CV MSE: 0.0691


--- LASSO Test Performance ---
RMSE: 0.2574
MAE:  0.1974
R²:   0.8658


C:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [22]:
# Compare the LASSO and MLR
#----------------------------
# 6. Extract Errors for the Wilcoxon Test
lasso_absolute_errors = np.abs(y_test - y_pred_lasso)

# (Optional) Run the Wilcoxon test right now against your MLR errors
from scipy.stats import wilcoxon
stat, p_value = wilcoxon(mlr_absolute_errors, lasso_absolute_errors)

print("\n--- Wilcoxon Test: MLR vs LASSO ---")
print(f"P-value: {p_value:.6f}")
if p_value < 0.05:
    print("Significant difference between MLR and LASSO.")
else:
    print("NO significant difference between MLR and LASSO.")


--- Wilcoxon Test: MLR vs LASSO ---
P-value: 0.961181
NO significant difference between MLR and LASSO.


In [23]:
print(lasso_absolute_errors.max())

2.2094011476014934


### Train RF 

In [24]:
# Set the preprocessing

tree_preprocessor = ColumnTransformer(
    transformers=[
        # 'passthrough' tells it to leave numeric features exactly as they are
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# set the experiment
rf_pipeline = Pipeline(steps=[
    ('preprocessor', tree_preprocessor),
    ('model', RandomForestRegressor(random_state=SEED))
])
rf_pipeline.fit(X_train, y_train)
joblib.dump(rf_pipeline, "best_rf_model.pkl")

['best_rf_model.pkl']

In [25]:

param_grid_rf = {
    'model__n_estimators': [200, 300, 500, 700],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': np.arange(2, 10, 1).tolist()
}

rf_cv = GridSearchCV(estimator=rf_pipeline, param_grid=param_grid_rf, 
                     cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

In [26]:
print("Tuning Random Forest (Unscaled)...")
rf_cv.fit(X_train, y_train)

best_rf = rf_cv.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_absolute_errors = np.abs(y_test - y_pred_rf)

print("=== Random Forest Performance ===")
print(f"Best Params: {rf_cv.best_params_}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_rf):.4f}")
print(f"R²:   {r2_score(y_test, y_pred_rf):.4f}\n")

Tuning Random Forest (Unscaled)...


C:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


=== Random Forest Performance ===
Best Params: {'model__max_depth': None, 'model__min_samples_split': 4, 'model__n_estimators': 700}
RMSE: 0.1743
MAE:  0.1148
R²:   0.9384



In [27]:
# Wilcoxon test 

stat_rf, p_rf = wilcoxon(mlr_absolute_errors, rf_absolute_errors)
print("=== Wilcoxon Tests (vs MLR Baseline) ===")
print(f"Random Forest p-value: {p_rf:.6f}")

=== Wilcoxon Tests (vs MLR Baseline) ===
Random Forest p-value: 0.000000


In [28]:
# Export the entire fitted pipeline (preprocessor + model)
joblib.dump(best_rf, 'best_rf_model.pkl')

joblib.dump(X_train.columns.tolist(), "model_columns.pkl")

['model_columns.pkl']

In [30]:
# Save the tuned pipeline (contains ColumnTransformer + Best RandomForest)
joblib.dump(best_rf, "best_rf_model.pkl")

# Save post-encoding feature names generated by ColumnTransformer
model_columns = best_rf.named_steps["preprocessor"].get_feature_names_out()
joblib.dump(list(model_columns), "model_columns.pkl")

# Save raw dataset for dashboard visuals and comparables
df.to_csv("df_cleaned_rf_trained.csv", index=False)

print("✅ Saved best_rf_model.pkl, model_columns.pkl, and df_cleaned_rf_traine.csv!")

✅ Saved best_rf_model.pkl, model_columns.pkl, and df_cleaned_rf_traine.csv!


### TRAIN AND FIT XGBoost

In [59]:
# set the preprocessing
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', tree_preprocessor),
    ('model', XGBRegressor(random_state=SEED, objective='reg:squarederror'))
])

param_grid_xgb = {
    'model__n_estimators': np.arange(200, 500, 100).tolist(),     # [200, 300, 400]
    'model__learning_rate': np.arange(0.01, 0.21, 0.05).tolist(),  # [0.01, 0.06, 0.11, 0.16]
    'model__max_depth': np.arange(4, 12, 2).tolist(),              # [2, 4, 6, 8]
    'model__subsample': np.arange(0.4, 1.1, 0.2).round(2).tolist() # [0.4, 0.6, 0.8, 1.0]
}

xgb_cv = GridSearchCV(estimator=xgb_pipeline, param_grid=param_grid_xgb, 
                      cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

In [60]:
print("Tuning XGBoost (Unscaled)...")
xgb_cv.fit(X_train, y_train)

best_xgb = xgb_cv.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
xgb_absolute_errors = np.abs(y_test - y_pred_xgb)

print("=== XGBoost Performance ===")
print(f"Best Params: {xgb_cv.best_params_}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_xgb):.4f}")
print(f"R²:   {r2_score(y_test, y_pred_xgb):.4f}\n")

Tuning XGBoost (Unscaled)...
=== XGBoost Performance ===
Best Params: {'model__learning_rate': 0.060000000000000005, 'model__max_depth': 10, 'model__n_estimators': 400, 'model__subsample': 0.8}
RMSE: 0.1752
MAE:  0.1177
R²:   0.9378



C:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [38]:
# WILCOXON TESTS VS MLR BASELINE
stat_xgb, p_xgb = wilcoxon(mlr_absolute_errors, xgb_absolute_errors)
print(f"Random Forest p-value: {p_rf:.6f}")

Random Forest p-value: 0.000000


In [61]:
np.arange(0.01, 0.21, 0.05).tolist()

[0.01, 0.060000000000000005, 0.11, 0.16000000000000003]

### Train ANN

In [39]:
# 1. Build the ANN pipeline
# IMPORTANT: Using the ORIGINAL preprocessor to ensure numeric features are SCALED
ann_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # early_stopping=True prevents overfitting by holding out 10% of training data internally
    ('model', MLPRegressor(random_state=SEED, max_iter=2000, early_stopping=True))
])

In [40]:
# 2. Define hyperparameter grid
# These are standard starting ranges for tabular data ANNs
param_grid_ann = {
    'model__hidden_layer_sizes': [(50,), (100,), (50, 50)], # Single and dual layer architectures
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.01], # L2 penalty (regularization)
    'model__learning_rate_init': [0.001, 0.01]
}

# 3. Set up 5-Fold Cross-Validation
ann_cv = GridSearchCV(
    estimator=ann_pipeline,
    param_grid=param_grid_ann,
    cv=5, 
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

In [42]:
# 4. Run Hyperparameter Tuning
print("Tuning ANN hyperparameters using 5-Fold CV (This may take a while)...")
ann_cv.fit(X_train, y_train)

# 5. Extract best parameters and CV MSE
print(f"\nBest ANN Parameters: {ann_cv.best_params_}")
print(f"Lowest CV MSE: {-ann_cv.best_score_:.4f}")

Tuning ANN hyperparameters using 5-Fold CV (This may take a while)...

Best ANN Parameters: {'model__activation': 'tanh', 'model__alpha': 0.0001, 'model__hidden_layer_sizes': (50, 50), 'model__learning_rate_init': 0.01}
Lowest CV MSE: 0.0549


In [43]:
# 6. Evaluate best refitted ANN on 30% Test Set
best_ann_model = ann_cv.best_estimator_
y_pred_ann = best_ann_model.predict(X_test)
ann_absolute_errors = np.abs(y_test - y_pred_ann)

print("=== ANN Test Performance ===")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ann)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_ann):.4f}")
print(f"R²:   {r2_score(y_test, y_pred_ann):.4f}\n")

=== ANN Test Performance ===
RMSE: 0.2428
MAE:  0.1716
R²:   0.8806



C:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [44]:
stat_ann, p_value_ann = wilcoxon(mlr_absolute_errors, ann_absolute_errors)

print("=== Wilcoxon Signed-Rank Test: MLR vs ANN ===")
print(f"Wilcoxon Statistic: {stat_ann:.4f}")
print(f"P-value: {p_value_ann:.6f}")

if p_value_ann < 0.05:
    print("Conclusion: The difference between MLR and ANN is statistically significant (p < 0.05).")
else:
    print("Conclusion: The difference between MLR and ANN is NOT statistically significant.")

=== Wilcoxon Signed-Rank Test: MLR vs ANN ===
Wilcoxon Statistic: 8057160.0000
P-value: 0.000000
Conclusion: The difference between MLR and ANN is statistically significant (p < 0.05).


In [48]:
# 2. Run the Wilcoxon Signed-Rank Test (RF vs XGBoost)
stat_rf_xgb, p_val_rf_xgb = wilcoxon(rf_absolute_errors, xgb_absolute_errors)

print("=== Wilcoxon Signed-Rank Test: Random Forest vs XGBoost ===")
print(f"Wilcoxon Statistic: {stat_rf_xgb:.4f}")
print(f"P-value: {p_val_rf_xgb:.6f}")

# 3. Interpret the statistical result
if p_val_rf_xgb < 0.05:
    print("Conclusion: The difference in performance between Random Forest and XGBoost IS statistically significant (p < 0.05).")
else:
    print("Conclusion: The difference in performance between Random Forest and XGBoost is NOT statistically significant (p >= 0.05).")

=== Wilcoxon Signed-Rank Test: Random Forest vs XGBoost ===
Wilcoxon Statistic: 9575998.5000
P-value: 0.000000
Conclusion: The difference in performance between Random Forest and XGBoost IS statistically significant (p < 0.05).


### SHAP 

In [65]:
# save train data

joblib.dump(X_train.columns.tolist(), "model_columns.pkl")

['model_columns.pkl']